# Spark Exercises 06 — Nested & Semi-structured Data

This is where Spark leaves a flat pandas/Polars table behind. Real event data arrives as **nested JSON**: a `device` object, a `geo` object, and an `items` **array** of products. Spark reads this natively into `struct` and `array` columns and gives you first-class tools — dot access, `explode`, `from_json` — to work with it without ever flattening to messy strings.

**Data file:** `data/events.jsonl` — 2,500 clickstream events (JSON Lines).

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### 2. Read the JSON Lines file with `spark.read.json("data/events.jsonl")` into `events`, then `printSchema()`. Notice the **nested structs** (`device`, `geo`) and the **array** (`items`).

### 3. Show 2 events **vertically** so you can see the nested structure rendered.

### 4. **Dot access.** Reach into the structs: select `event_id`, `event_type`, `device.os`, `device.is_mobile`, `geo.country`. Show 8 rows.

### 5. Count events **per operating system** (`device.os`), most common first.

### 6. **Flatten a struct.** Use `select("device.*")` to expand all of `device`'s fields into top-level columns. Show 5 rows.

### 7. **Filter on nested fields.** Keep only events from mobile devices using the `Chrome` browser. How many are there?

### 8. The `items` array is only present for `add_to_cart` / `purchase` events. Add a column `n_items = size(items)` and show it for 8 events (events without items show `-1`).

### 9. **Explode the array.** Take only `purchase` events, then `explode` `items` so each product in a basket becomes its **own row**. Select `event_id`, `user_id`, `item.sku`, `item.qty`, `item.price`. Show 10.

### 10. From the exploded `lines`, compute total **purchase revenue** (`sum(qty * price)`), rounded to 2 decimals.

### 11. Which **5 SKUs** were purchased most often (count of exploded item rows)?

### 12. **`posexplode`** also gives the position within the array. On `purchases` use `posexplode(items)` to get `pos` and `item`, and show the first item (`pos == 0`) of 8 baskets.

### 13. **Build and parse JSON.** Turn the `device` struct back into a JSON string with `F.to_json`, then parse it back into a struct with `F.from_json` using an explicit schema. Show `os` and `browser` from the re-parsed struct.